<a href="https://colab.research.google.com/github/neelameghana192311002/CSA6301---Threat-Intelligence-and-Network-Security/blob/main/UNIT4LAB/Exercise_3_Signature_Based_IDS_Detection_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
IDS_RULES = [
    {
        "sid": 2000001,
        "msg": "Possible SQL Injection",
        "proto": "tcp",
        "dst_port": 80,
        "content": "union select"
    },
    {
        "sid": 2000002,
        "msg": "Directory Traversal Attempt",
        "proto": "tcp",
        "dst_port": 80,
        "content": "../../../etc/passwd"
    },
    {
        "sid": 2000003,
        "msg": "Suspicious RDP Brute Force Pattern",
        "proto": "tcp",
        "dst_port": 3389,
        "content": "login_attempt"
    },
]


def scan_packet(packet, rules=IDS_RULES):
    """
    packet: {"proto", "dst_port", "payload"}.

    A rule fires only if BOTH the protocol/port match
    AND the content pattern is found (case-insensitive).
    """
    alerts = []

    payload_lower = packet["payload"].lower()

    for rule in rules:
        if (
            packet["proto"] == rule["proto"]
            and packet["dst_port"] == rule["dst_port"]
        ):
            if rule["content"] in payload_lower:
                alerts.append(rule["msg"])

    return alerts

In [2]:
def test_experiment3():
    sqli_packet = {
        "proto": "tcp",
        "dst_port": 80,
        "payload": "id=1' UNION SELECT user,pass FROM accounts--"
    }

    assert "Possible SQL Injection" in scan_packet(sqli_packet)

    traversal_packet = {
        "proto": "tcp",
        "dst_port": 80,
        "payload": "GET /files?path=../../../etc/passwd"
    }

    assert "Directory Traversal Attempt" in scan_packet(traversal_packet)

    # Same content string but wrong port context -> must NOT alert
    wrong_context = {
        "proto": "tcp",
        "dst_port": 22,
        "payload": "id=1' UNION SELECT user,pass FROM accounts--"
    }

    assert scan_packet(wrong_context) == []

    print("All test cases passed.")


test_experiment3()

All test cases passed.
